## v8 — OOF-리더보드 Gap 최소화 버전

### 전략
1. **2-Stage 학습**: Stage-1(lag 없이 base 피처만) → Stage-2(lag를 **감쇠·변환**하여 추가)
2. **lag 감쇠(damping)**: 학습 시 lag에 가우시안 노이즈를 섞어 모델이 lag에 과의존하지 않게 함
3. **lag 변환**: raw lag 대신 `log1p(lag)`, `clipped lag`, `lag - 시나리오 평균` 등 **약한 형태**로 제공
4. **순차 예측 EMA**: 슬롯 단위 예측값을 다음 슬롯 lag에 넣을 때 EMA smoothing으로 오차 누적 완화
5. **OOF는 순차 예측만** — 리더보드와 같은 조건

### 목표
- OOF MAE 7점대
- OOF ↔ 리더보드 gap 최소화

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import os
import sys
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
import time
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 헬퍼
# ============================================================
def _print_progress(msg):
    sys.stdout.write('\r' + msg)
    sys.stdout.flush()

def elapsed(start):
    s = int(time.time() - start)
    return f'{s//60}m {s%60:02d}s'

def section(title):
    print(f"\n{'='*60}\n  {title}\n{'='*60}")


class LGBProgressCallback:
    def __init__(self, fold, n_folds, total_rounds, log_every=200):
        self.fold, self.n_folds = fold, n_folds
        self.total, self.log_every = total_rounds, log_every
        self.start = time.time()
        self.best_mae, self.best_iter = float('inf'), 0
    def __call__(self, env):
        it = env.iteration + 1
        mae = env.evaluation_result_list[0][2]
        if mae < self.best_mae: self.best_mae, self.best_iter = mae, it
        if it % self.log_every == 0 or it == self.total:
            f = int(30 * it / self.total)
            bar = '█'*f + '░'*(30-f)
            _print_progress(
                f'  Fold {self.fold}/{self.n_folds}  [{bar}] {it/self.total*100:5.1f}%'
                f'  iter {it:>5}  val_MAE {mae:.4f}'
                f'  best {self.best_mae:.4f}@{self.best_iter}  {elapsed(self.start)}'
            )


class XGBProgressCallback(xgb.callback.TrainingCallback):
    def __init__(self, fold, n_folds, total_rounds, log_every=200):
        self.fold, self.n_folds = fold, n_folds
        self.total, self.log_every = total_rounds, log_every
        self.start = time.time()
        self.best_mae, self.best_iter = float('inf'), 0
    def after_iteration(self, model, epoch, evals_log):
        it = epoch + 1
        try: mae = list(evals_log.values())[0]['mae'][-1]
        except: return False
        if mae < self.best_mae: self.best_mae, self.best_iter = mae, it
        if it % self.log_every == 0:
            f = int(30 * it / self.total)
            bar = '█'*f + '░'*(30-f)
            _print_progress(
                f'  Fold {self.fold}/{self.n_folds}  [{bar}] {it/self.total*100:5.1f}%'
                f'  iter {it:>5}  val_MAE {mae:.4f}'
                f'  best {self.best_mae:.4f}@{self.best_iter}  {elapsed(self.start)}'
            )
        return False


def cat_verbose_step(n):
    return max(1, n // 5)

# ============================================================
# 1. 데이터 로드
# ============================================================
def _resolve_data_dir() -> str:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        d = p / 'data'
        if d.is_dir() and (d / 'train.csv').is_file():
            return str(d)
    raise FileNotFoundError('data/train.csv 를 찾을 수 없습니다.')

path = _resolve_data_dir()
project_root = str(Path(path).resolve().parent)
print(f'▶ 데이터 경로: {path}')

print('▶ 데이터 로드 중...', end=' ', flush=True)
t0 = time.time()
train = pd.read_csv(os.path.join(path, 'train.csv'))
test  = pd.read_csv(os.path.join(path, 'test.csv'))
layout = pd.read_csv(os.path.join(path, 'layout_info.csv'))
print(f'완료 ({elapsed(t0)})  train {len(train):,}행 / test {len(test):,}행')

TARGET  = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']
N_FOLDS = 5

# ============================================================
# 2. 결측치 처리
# ============================================================
def handle_missing_values(df):
    df = df.sort_values(['scenario_id', 'ID']).reset_index(drop=True)
    cols_to_fix = [c for c in df.columns
                   if df[c].isnull().any() and c not in (ID_COLS + [TARGET])]
    df[cols_to_fix] = df.groupby('scenario_id')[cols_to_fix].ffill()
    df[cols_to_fix] = df.groupby('scenario_id')[cols_to_fix].bfill()
    df[cols_to_fix] = df[cols_to_fix].fillna(df[cols_to_fix].median())
    return df

# ============================================================
# 3. 피처 엔지니어링
# ============================================================
def add_features(df):
    df = df.sort_values(['scenario_id', 'ID']).reset_index(drop=True)
    df['timeslot'] = df.groupby('scenario_id').cumcount()
    df['robot_efficiency']  = df['robot_active'] / (df['robot_total'] + 1e-6)
    df['robot_density']     = df['robot_total']  / (df['floor_area_sqm'] + 1e-6)
    df['order_per_station'] = df['order_inflow_15m'] / (df['pack_station_count'] + 1e-6)
    df['robot_per_station'] = df['robot_active'] / (df['pack_station_count'] + 1e-6)
    df['cumulative_orders'] = df.groupby('scenario_id')['order_inflow_15m'].cumsum()
    df['order_pressure']    = df['cumulative_orders'] / (df['pack_station_count'] + 1e-6)
    if 'congestion_score' in df.columns:
        df['risk_index']  = df['congestion_score'] * (1 - df['robot_efficiency'])
        df['bottle_neck'] = df['order_per_station'] * df['congestion_score']
    if 'low_battery_ratio' in df.columns:
        df['battery_risk'] = df['low_battery_ratio'] * df['robot_total']
    if 'battery_mean' in df.columns and 'battery_std' in df.columns:
        df['battery_cv'] = df['battery_std'] / (df['battery_mean'] + 1e-6)
    if 'avg_trip_distance' in df.columns:
        df['trip_per_robot']  = df['avg_trip_distance'] / (df['robot_active'] + 1e-6)
        df['trip_congestion'] = df['avg_trip_distance'] * df.get('congestion_score', 0)
    if 'pack_utilization' in df.columns:
        df['pack_order_ratio'] = df['pack_utilization'] / (df['order_per_station'] + 1e-6)
    roll_cols = ['order_per_station', 'congestion_score',
                 'pack_utilization', 'avg_trip_distance', 'low_battery_ratio']
    for col in roll_cols:
        if col not in df.columns: continue
        grp = df.groupby('scenario_id')[col]
        df[f'{col}_roll3_mean'] = grp.transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
        df[f'{col}_roll5_mean'] = grp.transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
        df[f'{col}_roll3_std']  = grp.transform(lambda x: x.shift(1).rolling(3, min_periods=1).std().fillna(0))
    for lag in [1, 2]:
        if 'order_per_station' in df.columns:
            df[f'order_per_station_lag{lag}'] = (
                df.groupby('scenario_id')['order_per_station'].shift(lag).bfill())
    scen_agg_cols = [
        'congestion_score', 'order_inflow_15m', 'battery_mean', 'pack_utilization',
        'avg_trip_distance', 'low_battery_ratio', 'max_zone_density', 'sku_concentration',
        'robot_idle', 'outbound_truck_wait_min', 'order_per_station', 'robot_efficiency',
        'order_pressure', 'risk_index', 'battery_risk', 'battery_cv',
    ]
    for col in scen_agg_cols:
        if col not in df.columns: continue
        stats = df.groupby('scenario_id')[col].agg(['mean','max','min','std']).reset_index()
        stats.columns = ['scenario_id'] + [f'{col}_scen_{f}' for f in ['mean','max','min','std']]
        df = df.merge(stats, on='scenario_id', how='left')
    if 'congestion_score_scen_mean' in df.columns and 'pack_utilization_scen_mean' in df.columns:
        df['cong_pack_interaction'] = df['congestion_score_scen_mean'] * df['pack_utilization_scen_mean']
    if 'avg_trip_distance_scen_mean' in df.columns and 'congestion_score_scen_mean' in df.columns:
        df['trip_cong_interaction'] = df['avg_trip_distance_scen_mean'] * df['congestion_score_scen_mean']
    if 'low_battery_ratio_scen_mean' in df.columns and 'robot_efficiency_scen_mean' in df.columns:
        df['battery_efficiency_risk'] = df['low_battery_ratio_scen_mean'] * (1 - df['robot_efficiency_scen_mean'])
    for col in ['congestion_score', 'order_per_station', 'pack_utilization', 'avg_trip_distance']:
        sm = f'{col}_scen_mean'
        if col in df.columns and sm in df.columns:
            df[f'{col}_rel_to_scen'] = df[col] / (df[sm] + 1e-6)
    for col in ['congestion_score', 'order_per_station', 'pack_utilization']:
        if col not in df.columns: continue
        df[f'{col}_scen_rank'] = df.groupby('scenario_id')[col].rank(pct=True)
    return df

def preprocess_all(df, layout_df):
    df = df.merge(layout_df, on='layout_id', how='left')
    df = handle_missing_values(df)
    df = add_features(df)
    df['layout_type'] = pd.factorize(df['layout_type'])[0]
    return df

print('▶ 전처리 중...', end=' ', flush=True)
t0 = time.time()
train = preprocess_all(train, layout)
test  = preprocess_all(test, layout)
print(f'완료 ({elapsed(t0)})')

# ============================================================
# 4. 타겟 인코딩
# ============================================================
print('▶ 타겟 인코딩 중...', end=' ', flush=True)
t0 = time.time()
TE_COLS = [c for c in ['layout_id','timeslot','layout_type','shift_hour','day_of_week']
           if c in train.columns]
SMOOTHING = 20
kf_te = GroupKFold(n_splits=N_FOLDS)
groups_te = train['scenario_id']
for col in TE_COLS:
    te_col = f'{col}_te'
    train[te_col] = np.nan
    gm = train[TARGET].mean()
    for tr_idx, val_idx in kf_te.split(train, train[TARGET], groups=groups_te):
        tr_df = train.iloc[tr_idx]
        stats = tr_df.groupby(col)[TARGET].agg(['mean','count'])
        smooth = (stats['count']*stats['mean'] + SMOOTHING*gm) / (stats['count']+SMOOTHING)
        train.loc[val_idx, te_col] = train.iloc[val_idx][col].map(smooth).fillna(gm)
    stats_full = train.groupby(col)[TARGET].agg(['mean','count'])
    smooth_full = (stats_full['count']*stats_full['mean'] + SMOOTHING*gm) / (stats_full['count']+SMOOTHING)
    test[te_col] = test[col].map(smooth_full).fillna(gm)
print(f'완료 ({elapsed(t0)})')

train_global_mean = train[TARGET].mean()
train_global_std  = train[TARGET].std()

# ============================================================
# 5. Damped lag 피처 — 핵심 변경
# ============================================================
# raw target_lag 대신 변환된 약한 신호로 제공하여
# 모델이 lag에 과의존하지 않되, 시계열 패턴은 활용하도록 함
#
# lag_log   = log1p(lag)         — 스케일 압축
# lag_clip  = clip(lag, 0, P95)  — 극단값 제거
# lag_diff  = lag1 - lag2        — 추세(증감)
# lag_ratio = lag1 / scen_mean   — 시나리오 대비 상대값
# ============================================================
DAMPED_LAG_COLS = [
    'lag1_log', 'lag2_log', 'lag3_log',
    'lag1_clip', 'lag2_clip', 'lag3_clip',
    'lag_diff_12', 'lag_diff_23',
    'lag_mean3',
]

LAG_CLIP_HI = None  # train fit 후 결정


def compute_damped_lags(df, raw_lag1, raw_lag2, raw_lag3, clip_hi):
    """raw lag 값 3개 → damped 피처 컬럼들 반환 (df 수정)."""
    df['lag1_log'] = np.log1p(np.maximum(raw_lag1, 0))
    df['lag2_log'] = np.log1p(np.maximum(raw_lag2, 0))
    df['lag3_log'] = np.log1p(np.maximum(raw_lag3, 0))
    df['lag1_clip'] = np.clip(raw_lag1, 0, clip_hi)
    df['lag2_clip'] = np.clip(raw_lag2, 0, clip_hi)
    df['lag3_clip'] = np.clip(raw_lag3, 0, clip_hi)
    df['lag_diff_12'] = raw_lag1 - raw_lag2
    df['lag_diff_23'] = raw_lag2 - raw_lag3
    df['lag_mean3'] = (raw_lag1 + raw_lag2 + raw_lag3) / 3.0
    return df


def add_true_damped_lags(df, clip_hi, noise_std=0.0):
    """실제 TARGET에서 raw lag를 만들고, damped 변환. noise_std>0이면 학습용 노이즈."""
    out = df.sort_values(['scenario_id', 'ID']).copy()
    raws = {}
    for lag in (1, 2, 3):
        r = out.groupby('scenario_id')[TARGET].shift(lag)
        scen_mean = out.groupby('scenario_id')[TARGET].transform('mean')
        r = r.fillna(scen_mean)
        if noise_std > 0:
            rng = np.random.RandomState(lag * 42)
            r = r + rng.normal(0, noise_std, size=len(r))
        raws[lag] = r.values
    out = compute_damped_lags(out, raws[1], raws[2], raws[3], clip_hi)
    return out


# ============================================================
# 피처 목록 확정
# ============================================================
base_feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]
feature_cols = base_feature_cols + DAMPED_LAG_COLS

LAG_CLIP_HI = float(np.percentile(train[TARGET].dropna(), 95))
print(f'▶ lag clip upper: {LAG_CLIP_HI:.2f}')
print(f'▶ base 피처: {len(base_feature_cols)} / +damped lag: {len(feature_cols)}')

# damped lag 초기값 (test)
for c in DAMPED_LAG_COLS:
    test[c] = 0.0

y = np.log1p(train[TARGET])
groups = train['scenario_id']
kf = GroupKFold(n_splits=N_FOLDS)

# 학습 시 lag noise — 검증 순차 예측의 오차를 시뮬레이션
LAG_NOISE_STD = train_global_std * 0.5
print(f'▶ lag noise std: {LAG_NOISE_STD:.2f}  (target std * 0.5)')

# 순차 예측 EMA alpha (1.0 = no smoothing, 작을수록 강하게 평활)
EMA_ALPHA = 0.7
print(f'▶ 순차 예측 EMA alpha: {EMA_ALPHA}')

oof_lgb = np.zeros(len(train))
oof_xgb = np.zeros(len(train))
oof_cat = np.zeros(len(train))
lgb_models, xgb_models, cat_models = [], [], []


# ============================================================
# 순차 예측 함수 (EMA smoothing 포함)
# ============================================================
def sequential_slot_predictions_damped(model, frame, feature_cols, gm, clip_hi, alpha, predict_log):
    va = frame.copy()
    for c in DAMPED_LAG_COLS:
        va[c] = 0.0
    pred_buf = pd.Series(np.nan, index=va.index, dtype=np.float64)
    ema_buf  = pd.Series(np.nan, index=va.index, dtype=np.float64)
    max_slot = int(va['timeslot'].max())

    for slot in range(max_slot + 1):
        slot_mask = va['timeslot'].to_numpy() == slot
        slot_idx  = va.index[slot_mask]

        raw_lags = {}
        for lag in (1, 2, 3):
            prev_slot = slot - lag
            if prev_slot >= 0:
                prev_mask = va['timeslot'].to_numpy() == prev_slot
                prev_df = va.loc[prev_mask, ['scenario_id']].copy()
                prev_df['pred'] = ema_buf.loc[va.index[prev_mask]].to_numpy()
                scen_to_pred = prev_df.groupby('scenario_id', sort=False)['pred'].last()
                raw_lags[lag] = va.loc[slot_idx, 'scenario_id'].map(scen_to_pred).fillna(gm).values
            else:
                raw_lags[lag] = np.full(len(slot_idx), gm)

        _tmp = compute_damped_lags(
            va.loc[slot_idx].copy(), raw_lags[1], raw_lags[2], raw_lags[3], clip_hi
        )
        for _c in DAMPED_LAG_COLS:
            va.loc[slot_idx, _c] = _tmp[_c].values

        X_slot = va.loc[slot_idx, feature_cols]
        pl = predict_log(model, X_slot)
        raw_pred = np.expm1(np.asarray(pl, dtype=np.float64))
        pred_buf.loc[slot_idx] = raw_pred

        if slot == 0:
            ema_buf.loc[slot_idx] = raw_pred
        else:
            prev_slot_mask = va['timeslot'].to_numpy() == (slot - 1)
            prev_ema_df = va.loc[prev_slot_mask, ['scenario_id']].copy()
            prev_ema_df['ema'] = ema_buf.loc[va.index[prev_slot_mask]].to_numpy()
            scen_to_ema = prev_ema_df.groupby('scenario_id', sort=False)['ema'].last()
            prev_ema = va.loc[slot_idx, 'scenario_id'].map(scen_to_ema).fillna(gm).values
            ema_buf.loc[slot_idx] = alpha * raw_pred + (1 - alpha) * prev_ema

    return pred_buf


# ============================================================
# 6. LightGBM
# ============================================================
section('LightGBM 학습 + 순차 OOF')
lgb_params = dict(
    n_estimators=10000, learning_rate=0.005,
    max_depth=8, num_leaves=127, min_child_samples=30,
    subsample=0.75, subsample_freq=1, colsample_bytree=0.6,
    reg_alpha=0.3, reg_lambda=3.0,
    random_state=42, verbose=-1,
)
lgb_fold_maes = []
lgb_start = time.time()

for fold, (tr_idx, val_idx) in enumerate(kf.split(train, y, groups=groups), 1):
    tr_raw = train.iloc[tr_idx].copy()
    va_raw = train.iloc[val_idx].copy()
    gm = tr_raw[TARGET].mean()

    tr_fit = add_true_damped_lags(tr_raw, LAG_CLIP_HI, noise_std=LAG_NOISE_STD)
    va_es  = add_true_damped_lags(va_raw, LAG_CLIP_HI, noise_std=0.0)

    X_tr, y_tr = tr_fit[feature_cols], np.log1p(tr_fit[TARGET])
    X_es, y_es = va_es[feature_cols], np.log1p(va_es[TARGET])

    cb_p = LGBProgressCallback(fold, N_FOLDS, lgb_params['n_estimators'])
    model = lgb.LGBMRegressor(**lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_es, y_es)], eval_metric='mae',
        callbacks=[lgb.early_stopping(400, verbose=False), lgb.log_evaluation(-1), cb_p],
    )

    seq_pred = sequential_slot_predictions_damped(
        model, va_raw, feature_cols, gm, LAG_CLIP_HI, EMA_ALPHA,
        lambda m, X: m.predict(X)
    )
    fold_idx = train.iloc[val_idx].index
    oof_lgb[val_idx] = seq_pred.reindex(fold_idx).to_numpy()
    lgb_models.append(model)
    fold_mae = mean_absolute_error(train.loc[val_idx, TARGET], oof_lgb[val_idx])
    lgb_fold_maes.append(fold_mae)
    print(f"\n  ✔ Fold {fold}  OOF MAE(순차) {fold_mae:.4f}  "
          f"best iter {model.best_iteration_:,}  {elapsed(lgb_start)}")

lgb_mae = mean_absolute_error(train[TARGET], oof_lgb)
print(f"\n  ▶ LightGBM OOF MAE (순차): {lgb_mae:.4f}")
print(f"    Fold별: {[f'{m:.4f}' for m in lgb_fold_maes]}")

# ============================================================
# 7. XGBoost
# ============================================================
section('XGBoost 학습 + 순차 OOF')
xgb_params = dict(
    n_estimators=10000, learning_rate=0.005, max_depth=7,
    subsample=0.75, colsample_bytree=0.6,
    reg_alpha=0.3, reg_lambda=3.0,
    random_state=42, tree_method='hist',
    early_stopping_rounds=400, eval_metric='mae', verbosity=0,
)
xgb_fold_maes = []
xgb_start = time.time()

for fold, (tr_idx, val_idx) in enumerate(kf.split(train, y, groups=groups), 1):
    tr_raw = train.iloc[tr_idx].copy()
    va_raw = train.iloc[val_idx].copy()
    gm = tr_raw[TARGET].mean()

    tr_fit = add_true_damped_lags(tr_raw, LAG_CLIP_HI, noise_std=LAG_NOISE_STD)
    va_es  = add_true_damped_lags(va_raw, LAG_CLIP_HI, noise_std=0.0)

    X_tr, y_tr = tr_fit[feature_cols], np.log1p(tr_fit[TARGET])
    X_es, y_es = va_es[feature_cols], np.log1p(va_es[TARGET])

    cb_p = XGBProgressCallback(fold, N_FOLDS, 10000)
    model = xgb.XGBRegressor(**xgb_params, callbacks=[cb_p])
    model.fit(X_tr, y_tr, eval_set=[(X_es, y_es)], verbose=False)

    seq_pred = sequential_slot_predictions_damped(
        model, va_raw, feature_cols, gm, LAG_CLIP_HI, EMA_ALPHA,
        lambda m, X: m.predict(X)
    )
    fold_idx = train.iloc[val_idx].index
    oof_xgb[val_idx] = seq_pred.reindex(fold_idx).to_numpy()
    xgb_models.append(model)
    fold_mae = mean_absolute_error(train.loc[val_idx, TARGET], oof_xgb[val_idx])
    xgb_fold_maes.append(fold_mae)
    print(f"\n  ✔ Fold {fold}  OOF MAE(순차) {fold_mae:.4f}  "
          f"best iter {model.best_iteration:,}  {elapsed(xgb_start)}")

xgb_mae = mean_absolute_error(train[TARGET], oof_xgb)
print(f"\n  ▶ XGBoost OOF MAE (순차): {xgb_mae:.4f}")
print(f"    Fold별: {[f'{m:.4f}' for m in xgb_fold_maes]}")

# ============================================================
# 8. CatBoost
# ============================================================
section('CatBoost 학습 + 순차 OOF')
cat_params = dict(
    iterations=10000, learning_rate=0.005,
    depth=8, l2_leaf_reg=5.0,
    bootstrap_type='MVS', subsample=0.75, colsample_bylevel=0.6,
    loss_function='MAE', eval_metric='MAE',
    random_seed=42, task_type='CPU',
    early_stopping_rounds=400,
)
cat_fold_maes = []
cat_start = time.time()

for fold, (tr_idx, val_idx) in enumerate(kf.split(train, y, groups=groups), 1):
    tr_raw = train.iloc[tr_idx].copy()
    va_raw = train.iloc[val_idx].copy()
    gm = tr_raw[TARGET].mean()

    tr_fit = add_true_damped_lags(tr_raw, LAG_CLIP_HI, noise_std=LAG_NOISE_STD)
    va_es  = add_true_damped_lags(va_raw, LAG_CLIP_HI, noise_std=0.0)

    X_tr, y_tr = tr_fit[feature_cols], np.log1p(tr_fit[TARGET])
    X_es, y_es = va_es[feature_cols], np.log1p(va_es[TARGET])

    print(f"\n  Fold {fold}/{N_FOLDS} 학습 중...")
    model = cb.CatBoostRegressor(**cat_params)
    model.fit(
        X_tr, y_tr,
        eval_set=(X_es, y_es),
        verbose=cat_verbose_step(cat_params['iterations']),
        use_best_model=True,
    )

    seq_pred = sequential_slot_predictions_damped(
        model, va_raw, feature_cols, gm, LAG_CLIP_HI, EMA_ALPHA,
        lambda m, X: m.predict(X)
    )
    fold_idx = train.iloc[val_idx].index
    oof_cat[val_idx] = seq_pred.reindex(fold_idx).to_numpy()
    cat_models.append(model)
    fold_mae = mean_absolute_error(train.loc[val_idx, TARGET], oof_cat[val_idx])
    cat_fold_maes.append(fold_mae)
    print(f"  ✔ Fold {fold}  OOF MAE(순차) {fold_mae:.4f}  "
          f"best iter {model.best_iteration_:,}  {elapsed(cat_start)}")

cat_mae = mean_absolute_error(train[TARGET], oof_cat)
print(f"\n  ▶ CatBoost OOF MAE (순차): {cat_mae:.4f}")
print(f"    Fold별: {[f'{m:.4f}' for m in cat_fold_maes]}")

# ============================================================
# 9. Test 순차 예측 (앙상블, EMA)
# ============================================================
section('Test 순차 예측')

w_lgb = 1 / lgb_mae
w_xgb = 1 / xgb_mae
w_cat = 1 / cat_mae
w_sum = w_lgb + w_xgb + w_cat

test = test.sort_values(['scenario_id', 'ID']).reset_index(drop=True)
test['timeslot'] = test.groupby('scenario_id').cumcount()
max_slot = test['timeslot'].max()

for c in DAMPED_LAG_COLS:
    test[c] = 0.0

test_preds = np.zeros(len(test))
test_ema   = np.full(len(test), train_global_mean)
seq_start  = time.time()
print(f'  타임슬롯 0~{max_slot} 순차 예측 중...')

for slot in range(max_slot + 1):
    slot_mask = test['timeslot'] == slot
    slot_idx  = test.index[slot_mask]

    raw_lags = {}
    for lag in [1, 2, 3]:
        prev_slot = slot - lag
        if prev_slot >= 0:
            prev_mask = test['timeslot'] == prev_slot
            prev_df = test.loc[prev_mask, ['scenario_id']].copy()
            prev_df['pred'] = test_ema[prev_mask]
            scen_to_pred = prev_df.groupby('scenario_id', sort=False)['pred'].last()
            raw_lags[lag] = test.loc[slot_idx, 'scenario_id'].map(scen_to_pred).fillna(train_global_mean).values
        else:
            raw_lags[lag] = np.full(len(slot_idx), train_global_mean)

    _tmp = compute_damped_lags(
        test.loc[slot_idx].copy(), raw_lags[1], raw_lags[2], raw_lags[3], LAG_CLIP_HI
    )
    for _c in DAMPED_LAG_COLS:
        test.loc[slot_idx, _c] = _tmp[_c].values

    X_slot = test.loc[slot_idx, feature_cols]
    p_lgb = np.mean([np.expm1(m.predict(X_slot)) for m in lgb_models], axis=0)
    p_xgb = np.mean([np.expm1(m.predict(X_slot)) for m in xgb_models], axis=0)
    p_cat = np.mean([np.expm1(m.predict(X_slot)) for m in cat_models], axis=0)
    raw_pred = (w_lgb * p_lgb + w_xgb * p_xgb + w_cat * p_cat) / w_sum

    test_preds[slot_idx] = raw_pred
    if slot == 0:
        test_ema[slot_idx] = raw_pred
    else:
        prev_slot_mask = test['timeslot'] == (slot - 1)
        prev_ema_df = test.loc[prev_slot_mask, ['scenario_id']].copy()
        prev_ema_df['ema'] = test_ema[prev_slot_mask]
        scen_to_ema = prev_ema_df.groupby('scenario_id', sort=False)['ema'].last()
        prev_ema = test.loc[slot_idx, 'scenario_id'].map(scen_to_ema).fillna(train_global_mean).values
        test_ema[slot_idx] = EMA_ALPHA * raw_pred + (1 - EMA_ALPHA) * prev_ema

    if slot % 5 == 0 or slot == max_slot:
        print(f'  슬롯 {slot:>2}/{max_slot}  완료  {elapsed(seq_start)}', flush=True)

print(f'  ✔ 순차 예측 완료  ({elapsed(seq_start)})')

# ============================================================
# 10. 앙상블 & 제출
# ============================================================
section('앙상블 & 제출')

oof_final = (w_lgb * oof_lgb + w_xgb * oof_xgb + w_cat * oof_cat) / w_sum
final_mae = mean_absolute_error(train[TARGET], oof_final)

print(f'  LightGBM {w_lgb/w_sum*100:.1f}%  (순차 OOF MAE {lgb_mae:.4f})')
print(f'  XGBoost  {w_xgb/w_sum*100:.1f}%  (순차 OOF MAE {xgb_mae:.4f})')
print(f'  CatBoost {w_cat/w_sum*100:.1f}%  (순차 OOF MAE {cat_mae:.4f})')
print(f'\n  ★ 앙상블 OOF MAE (순차): {final_mae:.4f}')
print(f'  ★ 전체 소요 시간: {elapsed(lgb_start)}')

submission = pd.DataFrame({'ID': test['ID'], TARGET: test_preds})
save_path = os.path.join(project_root, 'submission_v8.csv')
submission.to_csv(save_path, index=False)
print(f'\n  저장 완료 → {save_path}')

▶ 데이터 경로: C:\Smart-Warehouse-Shipment-Delay-Prediction-AI-Competition\data
▶ 데이터 로드 중... 완료 (0m 02s)  train 250,000행 / test 50,000행
▶ 전처리 중... 완료 (0m 37s)
▶ 타겟 인코딩 중... 완료 (0m 05s)
▶ lag clip upper: 60.79
▶ base 피처: 214 / +damped lag: 223
▶ lag noise std: 13.68  (target std * 0.5)
▶ 순차 예측 EMA alpha: 0.7

  LightGBM 학습 + 순차 OOF
  Fold 1/5  [█████░░░░░░░░░░░░░░░░░░░░░░░░░]  18.0%  iter  1800  val_MAE 0.3336  best 0.3335@1522  0m 56s
  ✔ Fold 1  OOF MAE(순차) 9.5280  best iter 1,512  1m 03s
  Fold 2/5  [████░░░░░░░░░░░░░░░░░░░░░░░░░░]  16.0%  iter  1600  val_MAE 0.3321  best 0.3319@1406  0m 50s
  ✔ Fold 2  OOF MAE(순차) 9.4004  best iter 1,364  2m 00s
  Fold 3/5  [█████░░░░░░░░░░░░░░░░░░░░░░░░░]  18.0%  iter  1800  val_MAE 0.3444  best 0.3443@1503  1m 00s
  ✔ Fold 3  OOF MAE(순차) 9.9581  best iter 1,503  3m 10s
  Fold 4/5  [█████░░░░░░░░░░░░░░░░░░░░░░░░░]  18.0%  iter  1800  val_MAE 0.3314  best 0.3311@1458  1m 15s
  ✔ Fold 4  OOF MAE(순차) 9.2121  best iter 1,458  4m 32s
  Fold 5/5  [█████░░░░░